# How Blokus Arena is built

This notebook walks through the Blokus Arena code for someone new to it.
It follows the order the game was built in: the rules engine, the heuristic bots, the LLM players, and a seat for a person.
Every step runs the real code in `backend/src/blokus`, so what you see here is what the app runs.

To run it, install the project with `uv sync` and pick the project's `.venv` as the kernel (VS Code offers it).
In a browser, `uv run --with jupyterlab jupyter lab` works too.
Everything runs offline except one cell in part 3, which calls a real LLM if you have an API key in `.env`.
Games played here are not saved to the app's leaderboard.

| Part | What it covers |
|---|---|
| 1. The rules engine | Pieces, the board, legal moves, turns and seats |
| 2. Bot players | The player interface, a bot from scratch, Greedy and Tactician |
| 3. LLM players | Models, prompts, tools, a rehearsal with a scripted model, and a real model |
| 4. A human player | How a person joins a game, and a game you can play in this notebook |
| 5. After the game | Game records and ELO ratings |

In [ ]:
import asyncio
import inspect
import random

from dotenv import find_dotenv, load_dotenv
from IPython.display import SVG, Markdown, display

from blokus.providers import configure_tracing

load_dotenv(find_dotenv(usecwd=True))  # API keys, used only by the real-model cell in part 3
configure_tracing()  # Agents SDK traces go to OpenAI, so they stay off without an OpenAI key

A few helpers for this notebook: pictures of the board and the pieces, and a way to show source code.
The web app draws the board in `frontend/src/blokus_ui/render.py`.
These helpers are smaller and draw the same data: a board as rows of color names, the way a game snapshot holds it.

In [ ]:
import textwrap

from blokus.board import BOARD_SIZE, COLUMNS, EMPTY, NUM_CELLS, Board, Color, cell_name, parse_cell
from blokus.pieces import PIECES
from blokus.snapshot import color_key, grid_view

# The web app's colors (frontend/src/blokus_ui/render.py). None is an empty cell.
FILL = {
    "blue": "#2f6fde",
    "yellow": "#eab308",
    "red": "#dc2d45",
    "green": "#1f9d55",
    None: "#eef1f5",
}
PX = 20  # pixels per board cell


def tag(name, text="", **attributes):
    """An SVG element. Underscores in attribute names become dashes, as in font_size=9."""
    pairs = " ".join(f'{key.replace("_", "-")}="{value}"' for key, value in attributes.items())
    return f"<{name} {pairs}>{text}</{name}>"


def image(width, height, elements, font_size=10):
    svg = tag(
        "svg",
        "".join(elements),
        xmlns="http://www.w3.org/2000/svg",
        width=width,
        height=height,
        font_family="sans-serif",
        font_size=font_size,
        text_anchor="middle",
    )
    return SVG(svg)


def board_svg(grid, dots=None, outline=()):
    """Draw a board given as rows of color keys ("blue", ... or None), as a snapshot has it.

    `dots` maps cell indices to a color key, and `outline` is a set of cell indices to frame.
    """
    dots, side = dots or {}, PX * (BOARD_SIZE + 1)
    svg = [tag("rect", x=PX, y=PX, width=side - PX, height=side - PX, fill="#cbd2db")]
    for k in range(BOARD_SIZE):
        svg.append(tag("text", COLUMNS[k], x=PX * (k + 1.5), y=PX * 0.7, fill="#6b7280"))
        svg.append(tag("text", k + 1, x=PX * 0.5, y=PX * (k + 1.7), fill="#6b7280"))
    for i in range(NUM_CELLS):
        row, col = divmod(i, BOARD_SIZE)
        x, y = PX * (col + 1), PX * (row + 1)
        svg.append(tag("rect", x=x, y=y, width=PX - 1, height=PX - 1, fill=FILL[grid[row][col]]))
        if i in dots:
            svg.append(tag("circle", cx=x + PX / 2, cy=y + PX / 2, r=3.5, fill=FILL[dots[i]]))
    for i in outline:
        row, col = divmod(i, BOARD_SIZE)
        x, y = PX * (col + 1) + 1, PX * (row + 1) + 1
        frame = {"fill": "none", "stroke": "#111", "stroke_width": 2}
        svg.append(tag("rect", x=x, y=y, width=PX - 3, height=PX - 3, **frame))
    return image(side, side, svg, font_size=9)


def draw(board, *, corners=(), outline=()):
    """Draw a Board, with dots on the corner points of each color in `corners`."""
    dots = {i: color_key(color) for color in corners for i in board.anchors(color)}
    return board_svg(grid_view(board), dots, outline)


def draw_pieces(names, color="blue"):
    """Draw pieces by name, 11 to a row, with each name underneath.

    Pieces are drawn as `pieces.py` defines them: at most 5 squares wide and 3 tall.
    """
    px, per_row = 11, 11
    box_width, box_height = 6 * px, 3 * px + 26
    svg = []
    for k, name in enumerate(names):
        shape = PIECES[name].shape
        left, top = (k % per_row) * box_width, (k // per_row) * box_height
        indent = (4 - max(c for _, c in shape)) * px / 2  # center the piece in its box
        for r, c in shape:
            x, y = left + indent + c * px, top + r * px
            svg.append(tag("rect", x=x, y=y, width=px - 1, height=px - 1, fill=FILL[color]))
        svg.append(tag("text", name, x=left + 2.5 * px, y=top + 3 * px + 13, fill="#374151"))
    rows = -(-len(names) // per_row)
    return image(min(len(names), per_row) * box_width, rows * box_height, svg)


def show_source(obj):
    """Show the source code of a function or class from the project."""
    source = textwrap.dedent(inspect.getsource(obj)).rstrip()
    display(Markdown(f"```python\n{source}\n```"))

## 1. The rules engine

The engine is plain Python in three modules: `pieces.py`, `board.py` and `game.py`.
[BLOKUS.MD](BLOKUS.MD) has the rules.
Everything the players do later, including checking an LLM's or a person's move, goes through this code.

### Pieces

Each color has the same 21 pieces, from one square up to five.
`pieces.py` draws each one in ASCII, and works out its distinct rotations and reflections once, when it is imported.

In [ ]:
from blokus.pieces import TOTAL_SQUARES

print(f"{len(PIECES)} pieces with {TOTAL_SQUARES} squares in all")
print("Distinct orientations:", {name: len(piece.orientations) for name, piece in PIECES.items()})
print("Orientations in all:", sum(len(piece.orientations) for piece in PIECES.values()))
draw_pieces(list(PIECES))

Symmetric pieces have fewer orientations: X5 looks the same every way round, while F5 has 8.

### The board

`board.py` keeps the board as a list of 400 cells, numbered row by row from the top-left.
Players see cells as spreadsheet-style names: columns A-T and rows 1-20, so A1 is the top-left cell.
Each color starts in its own corner, and the colors move in the order Blue, Yellow, Red, Green.

In [ ]:
from blokus.board import START_CORNERS

print("K10 is cell", parse_cell("K10"), "and cell 189 is", cell_name(189))
print("Starting corners:", {color.label: cell_name(cell) for color, cell in START_CORNERS.items()})

The idea everything else builds on is the **corner point** (an *anchor* in the code).
A corner point is an empty cell that touches one of a color's pieces at a corner, but none of them along an edge.
Every new piece must cover one of its color's corner points.
Before a color's first move, its only corner point is its starting corner.

Move generation uses this: it places every orientation of every remaining piece so that one of its squares covers a corner point, then checks the other squares.
That takes about a millisecond, which matters because the bots look at every legal move on every turn.

In [ ]:
board = Board()
print("Blue's corner points:", [cell_name(i) for i in board.anchors(Color.BLUE)])
print("Blue's legal first moves:", len(board.legal_moves(Color.BLUE)))

move = board.validate(Color.BLUE, "V5", ["A1", "A2", "A3", "B3", "C3"])
board.apply(move)
print(
    f"After {move}, Blue's corner points are",
    *(cell_name(i) for i in sorted(board.anchors(Color.BLUE))),
)
print("and Blue has", len(board.legal_moves(Color.BLUE)), "legal moves.")
draw(board, corners=[Color.BLUE], outline=move.cells)

`Board.validate` turns a piece name and cell names into a `Move`, and raises `IllegalMove` for any rule the move breaks.
The message says why, in words meant for a player.
These are the exact messages an LLM player or a person gets when they try an illegal move.

In [ ]:
from blokus.board import IllegalMove

attempts = [
    ("Yellow's first piece, away from its corner", Color.YELLOW, "I2", ["A10", "A11"]),
    ("Blue touching its own piece along an edge", Color.BLUE, "I2", ["A4", "A5"]),
    ("Cells that don't make the piece named", Color.BLUE, "L4", ["D4", "E4", "F4", "G4"]),
    ("Blue away from its corner points", Color.BLUE, "I1", ["K10"]),
    ("A cell that isn't on the board", Color.BLUE, "I1", ["U4"]),
]
for label, color, piece, cells in attempts:
    try:
        board.validate(color, piece, cells)
    except IllegalMove as error:
        print(f"{label}:\n  {error}")

### Turns and seats

`game.py` adds turn order, who controls which color, and scoring.
With 2 players each player moves two colors.
With 3 players Green is shared: the players take turns moving it, and its score counts for nobody.
A color with no legal move is out for the rest of the game, since the board only fills up.
Each unplayed square costs a point, and playing all 21 pieces earns a bonus.

In [ ]:
from blokus.game import Game

for players in (2, 3, 4):
    game = Game(players)
    owners = {
        color.label: "shared" if game.owner(color) is None else f"player {game.owner(color)}"
        for color in Color
    }
    print(f"{players} players:", owners)

## 2. Bot players

### The player interface

Every player (a bot, an LLM or a person) implements one method, defined in `players/__init__.py`:

```python
async def choose_move(self, view: TurnView) -> Decision
```

A `TurnView` has everything a player may use: a private copy of the board, the color to move, the legal moves (never empty), which colors are its own and which are opponents, the standings, and the recent turns.
The player answers with a `Decision`: the move, plus optional reasoning, and a note when someone else chose the move for it.
`choose_move` is async so that an LLM can wait for an API and a person for a click, without holding up the rest of the app.

In [ ]:
from dataclasses import fields

from blokus.players import Decision, Player, TurnView

print("TurnView:", ", ".join(field.name for field in fields(TurnView)))
print("Decision:", ", ".join(field.name for field in fields(Decision)))

### A bot from scratch

A simple strategy that already beats random play: always play one of your biggest pieces.
Here it is as a complete player.
`kind` says what sort of player it is, and `detail` is a short description the app shows.

In [ ]:
class BiggestFirst(Player):
    """Plays one of its biggest pieces, at a random legal spot."""

    kind = "bot"

    def __init__(self, name, seed=None):
        super().__init__(name)
        self.rng = random.Random(seed)

    @property
    def detail(self):
        return "biggest piece, random spot"

    async def choose_move(self, view):
        biggest = max(len(move.cells) for move in view.legal_moves)
        return Decision(
            self.rng.choice([move for move in view.legal_moves if len(move.cells) == biggest])
        )

### Running a match

`Match` in `match.py` runs a game between players.
`Match.play()` is an async generator that yields an event after each step (`start`, `thinking`, `move`, `out` and `over`).
Each event carries a `GameSnapshot`: plain data about the whole game, which is all the web app ever draws.

Jupyter already runs an event loop, so this notebook uses `await` directly instead of `asyncio.run`.

In [ ]:
from blokus.match import Match
from blokus.players.heuristic import RandomPlayer

finished_games = []  # final snapshots, for the leaderboard in part 5


async def play_game(players):
    """Play a game to the end. Returns the match and its final snapshot."""
    match = Match(players)
    async for event in match.play():
        snapshot = event.snapshot
    finished_games.append(snapshot)
    return match, snapshot


def results(snapshot):
    """The scores and the winners of a finished game."""
    for player in snapshot.players:
        colors = " and ".join(color.label for color in player.colors)
        print(f"{player.name:<14} {colors:<16} score {player.score:>4}")
    if shared := snapshot.shared:
        print(
            f"{'(shared)':<14} {shared.label:<16} score {shared.score:>4}, which counts for nobody"
        )
    print("Winner:", " and ".join(snapshot.winners))


def describe(entry):
    """One line of a snapshot's move log."""
    color = entry.color.capitalize()
    if entry.piece is None:
        return f"Turn {entry.turn}: {color} has no legal move left, so it is out"
    note = f" ({entry.fallback})" if entry.fallback else ""
    cells = " ".join(entry.cells)
    return f"Turn {entry.turn}: {entry.player} played {color} {entry.piece} at {cells}{note}"


match, final = await play_game(
    [BiggestFirst("Biggest first", seed=1), RandomPlayer("Random", seed=2)]
)
results(final)
draw(match.game.board)

The snapshot's log has every turn, with who moved, and any reasoning or fallback note:

In [ ]:
for entry in final.log[:6]:
    print(describe(entry))

### Greedy

The two bots in the app are in `players/heuristic.py`.
Both score every legal move and play the best one, breaking ties at random.
`HeuristicPlayer` does that part, so each bot only supplies a `scorer` for the moves.
The scoring runs in a worker thread (`asyncio.to_thread`), so a slow turn doesn't freeze the web page.

In [ ]:
from blokus.players.heuristic import CENTRALITY, GreedyPlayer, HeuristicPlayer, TacticianPlayer

show_source(HeuristicPlayer.choose_move)
show_source(HeuristicPlayer.decide)

Greedy scores a move as 10 points per square plus 1.5 times the *centrality* of each square.
Centrality is 0 in the corners and 18 in the middle, so Greedy plays big pieces toward the center.
It never looks at its opponents.

In [ ]:
show_source(GreedyPlayer.scorer)
print("Centrality:", {name: CENTRALITY[parse_cell(name)] for name in ("A1", "E5", "J10", "T1")})

### Tactician

Tactician thinks in corner points, since they measure how much room a color has left.
For each move it adds up:

- `size`: squares placed, since unplayed squares cost points at the end.
- `mobility`: the corner points the move creates, minus the ones it uses up or blocks for itself.
- `block`: opponent corner points the piece covers.
- minus `friendly`: corner points of the player's other color that it covers (in a 2-player game).
- a pull toward the center for the first few pieces, which fades out after 6.

When it moves the shared color in a 3-player game, which scores for nobody, it only tries to block opponents.
The weights are in `TacticianWeights`, and were tuned with simulated games like the ones below.

In [ ]:
from blokus.players.heuristic import TacticianWeights

TacticianWeights()

To compare the two bots, here is a position 20 turns into a game.
`play_until` plays a match up to a turn and stops it, and `Match.turn_view()` then gives the view the next player would get.

In [ ]:
async def play_until(match, turns):
    """Play a match until `turns` turns are done, then stop it, to look at the position."""
    events = match.play()
    async for event in events:
        if event.snapshot.turn >= turns:
            break
    await events.aclose()


match = Match([TacticianPlayer("Tactician", seed=1), GreedyPlayer("Greedy", seed=2)])
await play_until(match, 20)
view = match.turn_view()
friendly = ", ".join(color.label for color in view.friendly)
opponents = ", ".join(color.label for color in view.opponents)
print(f"Turn {view.turn}: {view.color.label} to move, with {len(view.legal_moves)} legal moves.")
print(f"{view.color.label}'s player also moves {friendly}; the opponents are {opponents}.")

In [ ]:
tactician, greedy = match.players
for bot in (greedy, tactician):
    score = bot.scorer(view)
    print(bot.name)
    for move in sorted(view.legal_moves, key=score, reverse=True)[:5]:
        print(f"  {score(move):6.1f}  {move}  {bot.explain(view, move)}".rstrip())

best = max(view.legal_moves, key=tactician.scorer(view))
draw(view.board, corners=[*view.opponents, view.color], outline=best.cells)

The dots are corner points in each color's own color, and the framed piece is Tactician's pick.
Greedy ranks moves by size and position alone.
Tactician prefers moves that open new corner points for itself and cover its opponents' corner points.

### Bots against each other

`uv run blokus-sim` plays many bot games with the seats rotating, to check how the strategies compare.
Here is the same idea on a small scale: 20 rounds in which each pair plays once, with the players taking turns at playing Blue, which moves first.

In [ ]:
from collections import Counter

BOTS = {
    "Tactician": TacticianPlayer,
    "Greedy": GreedyPlayer,
    "Biggest first": BiggestFirst,
    "Random": RandomPlayer,
}


PAIRS = [("Tactician", "Greedy"), ("Greedy", "Biggest first"), ("Biggest first", "Random")]
wins = {pair: Counter() for pair in PAIRS}
for game in range(20):
    for first, second in PAIRS:
        seats = [BOTS[first](first, seed=game), BOTS[second](second, seed=game)]
        if game % 2:
            seats.reverse()
        _, snapshot = await play_game(seats)
        for winner in snapshot.winners:
            wins[first, second][winner] += 1 / len(snapshot.winners)  # a tie splits the win

for (first, second), won in wins.items():
    print(f"{first} {won[first]:g}, {second} {won[second]:g}")

## 3. LLM players

An LLM player (`players/llm.py`) is a `Player` like the bots, built on the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/).
Each turn it runs an agent with a prompt describing the position and two tools, and the engine checks every move the model submits.
This part builds one up in steps: choosing a model, the prompt, the tools, and the agent loop.

### Models and providers

Models are named `provider:model_id`, for example `gemini:gemini-3.1-flash-lite`.
`providers.py` maps each provider to an OpenAI-compatible API and the environment variable that holds its key.
So one client class, `AsyncOpenAI`, reaches all of them, including a local Ollama server.

In [ ]:
from blokus.providers import PROVIDERS, ModelSpec, available_presets

for provider in PROVIDERS.values():
    key = provider.api_key_env or "(no key)"
    print(f"{provider.name:<11} {key:<19} {provider.url() or '(the OpenAI default)'}")
print()
print(repr(ModelSpec.parse("openrouter:nvidia/nemotron-3.5-lightning:free")))
print("Presets you can use now:", available_presets())

`available_presets()` lists the preset models whose provider has a key set (or, for Ollama, a server running).
`build_model(spec)` turns a spec into an Agents SDK model, and `LLMPlayer.from_spec(name, spec)` builds a player around it.

### What the model is told

The agent's instructions, from `players/prompts.py`, are the same every turn.
They explain the coordinates, the placement rules, scoring, all 21 pieces as ASCII drawings, and how to use the tools.

In [ ]:
from blokus.players.llm import MAX_INVALID_ATTEMPTS, LLMPlayer
from blokus.players.prompts import system_prompt, turn_prompt

instructions = system_prompt(MAX_INVALID_ATTEMPTS)
pieces_start, pieces_end = instructions.index("# Pieces"), instructions.index("# Your turn")
print(instructions[:pieces_start])
print(f"[... the 21 pieces, {pieces_end - pieces_start:,} characters ...]\n")
print(instructions[pieces_end:])

Each turn's prompt describes the position from the mover's side.
It says which colors are the model's, draws the board as text with its corner points marked `*`, and lists its pieces with how many legal placements each has.
It ends with the standings, the recent moves, and the model's own reasoning from its last turn, as a short memory.

Here is a 3-player position where the LLM plays Red.
`BiggestFirst` sits in the LLM's seat while the position is set up.

In [ ]:
match = Match(
    [
        GreedyPlayer("Greedy", seed=1),
        TacticianPlayer("Tactician", seed=2),
        BiggestFirst("LLM 1", seed=3),
    ]
)
await play_until(match, 14)
view = match.turn_view()
print(turn_prompt(view, "LLM 1", notes="Head for the middle before Yellow closes it off."))

### The tools

The prompt doesn't list the legal moves: there can be hundreds.
The model gets two tools instead:

- `list_legal_moves(piece, corner_point)` lists exact legal placements for one piece, optionally only those covering one corner point.
- `place_piece(reasoning, piece, cells)` submits a move. The engine checks it with `Board.validate`, and if it is illegal the tool returns the reason, so the model can try again (up to 3 times).

Both are Python functions with the SDK's `@function_tool` decorator, which turns their signatures and docstrings into the tool descriptions the model sees.

In [ ]:
from blokus.players.llm import list_legal_moves, list_placements, place_piece

for tool in (list_legal_moves, place_piece):
    print(f"{tool.name}: {tool.description}")
    for name, schema in tool.params_json_schema["properties"].items():
        print(f"  {name} ({schema['type']}): {' '.join(schema['description'].split())}")
    print()

corner = cell_name(min(view.board.anchors(view.color)))
print(list_placements(view, "L5", corner_point=corner))

### The agent loop

`LLMPlayer.choose_move` runs a fresh agent each turn, with a per-turn state that the tools read and write: the view, the accepted move, and the number of illegal attempts.

- The run ends as soon as `place_piece` accepts a move, or the illegal attempts run out. A custom `tool_use_behavior` checks this after every tool call.
- If the model answers in plain text instead of calling a tool, it gets one reminder.
- If there is still no move, or the API fails, or 120 seconds pass, the player picks a random legal move and marks the `Decision` as a "random fallback". A slow or confused model can't stall the game.
- The model's reasoning is kept for its next turn's prompt.

In [ ]:
show_source(LLMPlayer.choose_move)

### A rehearsal with a scripted model

The Agents SDK's `ScriptedModel` plays back tool calls written in advance, without any API calls.
The tests use it, and it shows the loop clearly.
This one plays like a careless model: its first move is nowhere near its pieces, then it looks up the legal moves and places a piece.

In [ ]:
from agents.testing import ScriptedModel, assistant_message, function_call

board, anchors = view.board, view.board.anchors(view.color)
far = next(cell_name(i) for i in range(NUM_CELLS) if board.grid[i] == EMPTY and i not in anchors)
good = max(view.legal_moves, key=lambda move: len(move.cells))


def call(tool, number, **arguments):
    """One model reply: a single tool call."""
    return [function_call(tool, arguments, call_id=f"call-{number}")]


model = ScriptedModel(
    [
        call("place_piece", 1, reasoning="Take the open space.", piece="I1", cells=[far]),
        call("list_legal_moves", 2, piece=good.piece, corner_point=""),
        call(
            "place_piece",
            3,
            reasoning="A big piece at one of my corner points.",
            piece=good.piece,
            cells=good.cell_names,
        ),
    ]
)
decision = await LLMPlayer("LLM 1", model, "scripted").choose_move(view)
print(f"Played {decision.move} after {decision.invalid_attempts} illegal attempt.")
print("Reasoning:", decision.reasoning)
print("Fallback:", decision.fallback or "none")

outputs = [item for item in model.calls[1].input if item.get("type") == "function_call_output"]
print("\nWhat place_piece told the model after its first try:\n ", outputs[0]["output"])

And a model that only talks: it gets a reminder, still doesn't call a tool, and the player falls back to a random legal move.

In [ ]:
chatty = ScriptedModel(
    [[assistant_message("I'd play L5 near the middle.")], [assistant_message("Still thinking.")]]
)
decision = await LLMPlayer("LLM 2", chatty, "scripted", seed=0).choose_move(view)
print(f"{decision.fallback}: {decision.move}. Why: {decision.error}")

### A real model

This cell plays the same turn with a real model: the first preset you have an API key for, unless you set `MODEL` yourself.
It makes a few API calls with your key.
Free tiers often limit how many calls you can make per minute; when that happens the move shows up as a random fallback with the provider's error.

In [ ]:
presets = available_presets()
MODEL = (
    presets[0] if presets else None
)  # or e.g. "openrouter:openrouter/free"; None skips this cell

if MODEL is None:
    print("No LLM provider is set up, so this step is skipped. Add an API key to .env to try it.")
else:
    decision = await LLMPlayer.from_spec("LLM 1", MODEL).choose_move(view)
    print(f"{MODEL} played {decision.move}")
    print("Reasoning:", decision.reasoning or "(none)")
    print(f"Illegal attempts: {decision.invalid_attempts}, tokens used: {decision.tokens:,}")
    if decision.fallback:
        print(f"{decision.fallback}: {decision.error}")
    after = view.board.copy()
    after.apply(decision.move)
    display(draw(after, outline=decision.move.cells))

### A full game with an LLM

In the app, LLMs join the two bots through `lineup(guests)`, which seats Greedy and Tactician plus up to two guests and shuffles the seats.
A full game takes about 20 LLM turns, so this cell only runs if you turn it on.
The web app (`uv run blokus`) is a nicer way to watch one.

In [ ]:
from blokus.match import lineup

PLAY_FULL_GAME = False

if PLAY_FULL_GAME and MODEL:
    match = Match(lineup([LLMPlayer.from_spec("LLM 1", MODEL)], seed=1))
    async for event in match.play():
        if event.kind == "move" and event.snapshot.log[-1].player == "LLM 1":
            print(describe(event.snapshot.log[-1]))
    finished_games.append(event.snapshot)
    results(event.snapshot)
    display(draw(match.game.board))
else:
    print("Set PLAY_FULL_GAME = True to watch", MODEL or "an LLM", "play a full game.")

## 4. A human player

A person is one more `Player`, in `players/human.py`.
The difference is where the move comes from: `HumanPlayer.choose_move` waits for the front end to call `submit(turn, piece, cells)`.

In [ ]:
from blokus.players.human import HumanPlayer

show_source(HumanPlayer.choose_move)
show_source(HumanPlayer.submit)

- `choose_move` sets up a future and waits up to 15 seconds for it, plus one second so a last-moment click still arrives.
- `submit` checks the move with the same `Board.validate` as the LLM's `place_piece` tool. An illegal move returns the reason and the clock keeps running, so the person can try again.
- If time runs out, Greedy or Tactician (picked at random) moves instead, and the `Decision` says so, just like an LLM's random fallback.

### Adding a person to a game

The app adds a person the same way as an LLM: as a guest next to the two bots, with `lineup([HumanPlayer(), ...])`.
While a person is to move, the snapshot carries a `HumanTurn` with what a front end needs: the time limit, the remaining pieces, which of them fit somewhere, the corner points, and every legal placement.
Placements are text keys, such as `"I2:0,1"` for I2 on cells 0 and 1 (A1 and B1).
The web page uses them to preview pieces and snap them into place without asking the server.

Here nobody answers, so after the time limit a stand-in bot moves:

In [ ]:
nobody = HumanPlayer(time_limit=1, seed=4)
match = Match(lineup([nobody], shuffle=False, seed=4))  # Greedy, Tactician, then the person
events = match.play()
async for event in events:
    if turn := event.snapshot.human_turn:
        color = turn.color.capitalize()
        print(f"Turn {turn.turn}: the person moves {color}, with {turn.seconds:g}s on the clock.")
        print(f"{len(turn.playable)} of their {len(turn.pieces)} pieces fit somewhere.")
        print(f"There are {len(turn.legal)} legal placements, such as {turn.legal[0]}.")
        print("Corner points:", *(cell_name(row * BOARD_SIZE + col) for row, col in turn.anchors))
    if event.kind == "move" and event.snapshot.log[-1].player == nobody.name:
        print(event.snapshot.log[-1].error, describe(event.snapshot.log[-1]))
        break
await events.aclose()

### Play a game here

The next cell turns the notebook into a tiny front end.
The game runs in the background while you move with `await play(piece, cells)`.
You play Blue against Greedy (Yellow) and Tactician (Red).
Green is shared, and you move it first, then every third Green turn after that.
The clock is set to an hour, because typing a move takes longer than clicking one.

- `await your_turn()` shows the board, your corner points (dots) and the pieces that fit.
- `await play("V5", "A1 A2 A3 B3 C3")` places a piece: its name, then the cells it covers.
- `placements("L5")` lists legal spots for a piece, if you're stuck.
- `await autoplay()` plays the rest of your moves for you, biggest piece first.

These helpers only use the snapshots and `submit`, just like the web app.

In [ ]:
you = HumanPlayer("You", time_limit=3600)
match = Match([you, GreedyPlayer("Greedy", seed=5), TacticianPlayer("Tactician", seed=6)])
latest = None  # the newest snapshot
changed = asyncio.Event()


async def run_game():
    """Play the match in the background, keeping the newest snapshot in `latest`."""
    global latest
    try:
        async for event in match.play():
            latest = event.snapshot
            changed.set()
        finished_games.append(latest)
    finally:
        changed.set()  # wake up your_turn(), even if the game stopped with an error


async def your_turn(after=0, show=True):
    """Wait until you are to move on a turn later than `after`, or the game is over."""
    while latest is None or not (
        latest.over or (latest.human_turn and latest.human_turn.turn > after)
    ):
        if game_task.done():
            game_task.result()  # raises the error that stopped the game
        changed.clear()
        await changed.wait()
    if show:
        show_board(since=after)


def show_board(since=0):
    """Show the moves since turn `since` (or the last 3), the board, and your pieces."""
    entries = [e for e in latest.log if e.turn >= since] if since else latest.log[-3:]
    for entry in entries:
        print(describe(entry))
    last_move = {row * BOARD_SIZE + col for row, col in latest.last_move}
    turn = latest.human_turn
    if turn is None:
        results(latest)
        display(board_svg(latest.grid, outline=last_move))
        return
    shared = latest.shared and latest.shared.color == turn.color
    note = " (shared, so its score counts for nobody)" if shared else ""
    if entries:
        print()
    print(f"Turn {turn.turn}: your move, as {turn.color.capitalize()}{note}.")
    corner_points = {row * BOARD_SIZE + col: turn.color for row, col in turn.anchors}
    display(board_svg(latest.grid, corner_points, last_move))
    display(draw_pieces(turn.playable, turn.color))


def spot(key):
    """A placement key such as "I2:0,1" as (piece, cell names)."""
    piece, cells = key.split(":")
    return piece, [cell_name(int(i)) for i in cells.split(",")]


async def play(piece, cells, show=True):
    """Place `piece` on `cells`, written as "A1 A2" or ["A1", "A2"]."""
    turn = latest.human_turn
    if turn is None:
        print("The game is over." if latest.over else "It is not your turn.")
        return
    error = you.submit(turn.turn, piece, cells.split() if isinstance(cells, str) else cells)
    if error:
        print("Not allowed:", error)
        return
    await your_turn(after=turn.turn, show=show)


def placements(piece, limit=10):
    """Legal spots for one of your pieces."""
    name = piece.upper()
    spots = [cells for piece, cells in map(spot, latest.human_turn.legal) if piece == name]
    print(
        f"{len(spots)} legal placements for {name}"
        + (f", the first {limit}:" if len(spots) > limit else ":")
    )
    for cells in spots[:limit]:
        print(" ", " ".join(cells))


async def autoplay():
    """Play the rest of your moves: the biggest piece that fits, at its first legal spot."""
    await your_turn(show=False)
    while not latest.over:
        piece, cells = spot(max(latest.human_turn.legal, key=lambda key: key.count(",")))
        await play(piece, cells, show=False)
    await game_task
    show_board()


game_task = asyncio.create_task(run_game())
await your_turn()

A first try that misses the starting corner is refused with the reason, and it is still your turn:

In [ ]:
await play("I5", "B1 C1 D1 E1 F1")

In [ ]:
await play("V5", "A1 A2 A3 B3 C3")

Greedy and Tactician have moved, and now you move the shared Green from its corner, A20.

In [ ]:
placements("V5")

In [ ]:
await play("V5", "A18 A19 A20 B20 C20")

Keep going with `await play(...)`, or let `autoplay()` finish the game:

In [ ]:
await autoplay()

### How the web app does it

The web app runs the same kind of match, one per browser tab, in a Gradio event handler (`frontend/src/blokus_ui/app.py`).
It draws each snapshot as HTML, and while you are to move it sends the `HumanTurn` data to the page as JSON.
The script `frontend/src/blokus_ui/board.js` previews pieces, snaps them onto legal spots using the placement keys, and runs the 15-second clock.
When you click, it sends `{token, turn, piece, cells}` to the Python function `submit_move`, which calls `HumanPlayer.submit`: the same call as `play()` above.

To play there, run `uv run blokus` (or `scripts/start_mac.sh` to use Docker) and open the address it prints.

## 5. After the game

When a game ends, the app turns its final snapshot into a `GameRecord` (`records.py`) and adds it to the game history (`history.py`), which is saved under `data/games/`.
The leaderboard's ELO ratings (`ratings.py`) are recomputed from all the records, counting each game as a match between every pair of players.
Here is the leaderboard for the games in this notebook, kept in memory only:

In [ ]:
from blokus.history import GameHistory
from blokus.records import GameRecord

history = GameHistory(None)  # None keeps the games in memory instead of saving them
for snapshot in finished_games:
    history.add(GameRecord.from_snapshot(snapshot))
print(f"{len(history.records())} games")
for rank, rating in enumerate(history.ratings(), 1):
    print(f"{rank}. {rating.identity:<14} {rating.rating:5.0f}")

## Where to go next

- [ARCHITECTURE.MD](ARCHITECTURE.MD) explains how the modules fit together, and why.
- [README.md](README.md) covers running the app, with or without Docker, and hosting it on Hugging Face Spaces.
- `backend/tests` and `frontend/tests` show each part in small, exact examples.